# مشروع توقع أسعار العقارات باستخدام تعلم الآلة 🏠📊
مرحباً بك في هذا المشروع! الهدف هنا هو بناء وتدريب نموذج ذكاء اصطناعي قادر على توقع أسعار العقارات بناءً على مواصفاتها الأساسية (المساحة، عدد الغرف، وعدد دورات المياه).

---
## 1. استيراد المكتبات الأساسية
في هذه الخطوة، نقوم باستيراد المكتبات اللازمة للتعامل مع البيانات وبناء النموذج.

In [4]:
import pandas as pd           
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

## 2. تحميل البيانات وتجهيز النموذج
هنا نقوم بالخطوات التالية:
1. قراءة ملف البيانات `Aqar_data.csv` وتحديد الأعمدة الرقمية المهمة فقط.
2. فصل المتغيرات المستقلة (المميزات `x`) عن المتغير التابع (الهدف `y` وهو السعر).
3. تعريف نموذج **Random Forest Regressor** مع ضبط المعلمات (Hyperparameters) للحد من مشكلة الـ *Overfitting*.

In [11]:
df = pd.read_csv("Aqar_data.csv")
df['neighborhood'] = df['location'].str.replace('حي', '').str.replace('- الرياض', '').str.strip()


## 3. تقسيم البيانات وتقييم أداء النموذج
نقوم بتقسيم البيانات إلى مجموعتي تدريب واختبار بنسبة (80:20)، ثم ندرب النموذج ونحسب دقة التوقع ($R^2\ Score$) للتأكد من توازن الأداء وعدم وجود حفظ صم للبيانات (*Overfitting*).

In [ ]:
clean_df = df[
    (df['price'] < df['price'].quantile(0.98))
    & (df['size'] < df['size'].quantile(0.98))
].copy()


X = clean_df.drop(columns=['price'])
y = np.log1p(clean_df['price'])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train = X_train.copy()
X_test = X_test.copy()


m = 15
global_mean = y_train.mean()
stats = y_train.groupby(X_train['neighborhood']).agg(['count', 'mean'])
smooth = (stats['count'] * stats['mean'] + m * global_mean) / (
    stats['count'] + m
)

X_train['neighborhood_encoded'] = (
    X_train['neighborhood'].map(smooth).fillna(global_mean)
)
X_test['neighborhood_encoded'] = (
    X_test['neighborhood'].map(smooth).fillna(global_mean)
)

features = ['size', 'bedrooms', 'bathrooms', 'neighborhood_encoded']


model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=10,
    max_features=0.8,
    random_state=42,
)

model.fit(X_train[features], y_train)

train_score = model.score(X_train[features], y_train)
test_score = model.score(X_test[features], y_test)

print(f'دقة التدريب: {train_score:.4f}')
print(f'دقة الاختبار: {test_score:.4f}')

دقة التدريب: 0.7868
دقة الاختبار: 0.7432


## 4. نظام التوقع التفاعلي مع المستخدم 
في هذه الخطوة الأخيرة، نقوم بتدريب النموذج على كامل البيانات المتاحة لرفع دقة التوقع، ثم نتيح للمستخدم إدخال مواصفات العقار الذي يرغب في معرفة سعره بشكل تفاعلي ليقوم النموذج بحسابه فوراً.

In [18]:
m = 15
y_log_all = np.log1p(clean_df['price'])
global_mean_all = y_log_all.mean()

stats_all = y_log_all.groupby(clean_df['neighborhood']).agg(['count', 'mean'])
smooth_all = (stats_all['count'] * stats_all['mean'] + m * global_mean_all) / (stats_all['count'] + m)

clean_df_encoded = clean_df.copy()
clean_df_encoded['neighborhood_encoded'] = clean_df_encoded['neighborhood'].map(smooth_all).fillna(global_mean_all)

features = ['size', 'bedrooms', 'bathrooms', 'neighborhood_encoded']
X_full = clean_df_encoded[features]
y_full = y_log_all

final_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=4,
    min_samples_leaf=10,
    max_features=0.8,
    random_state=42
)
final_model.fit(X_full, y_full)

print("---------------- تم تدريب النموذج بنجاح ----------------")

# 2. إدخال بيانات العقار من المستخدم (مع الحي)
user_size = float(input("أدخل مساحة العقار من فضلك (رقم فقط): "))
user_beds = float(input("أدخل عدد غرف النوم (رقم فقط): "))
user_baths = float(input("أدخل عدد دورات المياه (رقم فقط): "))
user_neighborhood = input("أدخل اسم الحي: ")


user_encoded_nh = smooth_all.get(user_neighborhood, global_mean_all)

user_input = pd.DataFrame([[user_size, user_beds, user_baths, user_encoded_nh]], 
                          columns=features)

# 3. التوقع وإرجاع السعر للقيم الحقيقية
predict_log = final_model.predict(user_input)
predict_price = np.expm1(predict_log)[0]

print("-" * 40)
print(f"السعر المتوقع للعقار هو: {predict_price:,.2f} ريال")
print("تنويه: السعر مجرد توقع من النموذج قد يصيب ويخطئ")
print("-" * 40)



---------------- تم تدريب النموذج بنجاح ----------------
----------------------------------------
السعر المتوقع للعقار هو: 1,548,529.52 ريال
تنويه: السعر مجرد توقع من النموذج قد يصيب ويخطئ
----------------------------------------
